In [ ]:
#----------------------------------------------------------------------------------SETUP CODE------------------------------------------------------------------------------------

In [ ]:
"""
CELL 1 — Install dependencies
==============================
Runtime: T4 GPU
"""

!pip install -q trl transformers datasets accelerate bitsandbytes wandb

In [ ]:
"""
CELL 2 — Confirm T4 is available
"""


import subprocess, sys

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        "No GPU found. Go to Runtime → Change runtime type → T4 GPU and restart."
    )
print(result.stdout)

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU             : {torch.cuda.get_device_name(0)}")
print(f"VRAM total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
"""
CELL 3 — Connect to Weights & Biases
"""


import wandb

wandb.login()   # prompts for API key on first run

# Initialise the run — all training metrics will appear at wandb.ai/your-username
run = wandb.init(
    project="grpo-math-qwen",           # project name shown in wandb dashboard
    name="qwen2.5-1.5b-gsm8k-grpo",    # this specific experiment
    config={
        "model"     : "Qwen/Qwen2.5-1.5B-Instruct",
        "dataset"   : "openai/gsm8k",
        "algorithm" : "GRPO",
        "precision" : "int4 (bitsandbytes)",
        "task"      : "math reasoning",
    },
)
print(f"wandb run url: {run.url}")

In [ ]:
"""
CELL 4 — Load Qwen2.5-1.5B-Instruct in 4-bit precision

Why 4-bit?
  Qwen2.5-1.5B in float16  ≈ 3.0 GB  — fits, but leaves no headroom for
  GRPO's G=4 generations + gradients + optimizer states.
  In int4 (NF4)            ≈ 0.9 GB  — leaves ~13 GB free for all of the above.

Why NF4 over int4?
  NF4 (NormalFloat4) is information-theoretically optimal for normally-distributed
  weights (which neural network weights are). Better accuracy at the same bitwidth.

Why double_quant=True?
  Also quantises the quantisation constants themselves, saving a further ~0.4 GB
  with negligible quality loss.
"""


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit quantisation config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — best quality at 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16,  # compute in bf16, store in int4
    bnb_4bit_use_double_quant=True,     # quantise the quant constants too
)

print(f"Loading tokenizer from {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token      # needed for batched generation
tokenizer.padding_side = "left"                # decoder-only models pad on left

print(f"Loading model in 4-bit NF4...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",          # places layers on GPU automatically
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,     # required for Qwen architecture
)

# Freeze the base model — only the LoRA adapters (added by TRL) will train
model.config.use_cache = False              # disable KV cache during training
model.enable_input_require_grads()          # needed for gradient checkpointing

print("\nModel loaded successfully.")
print(f"  Dtype        : {next(model.parameters()).dtype}")
print(f"  Device       : {next(model.parameters()).device}")

# Memory report
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nVRAM usage:")
print(f"  Allocated : {allocated:.2f} GB")
print(f"  Reserved  : {reserved:.2f} GB")
print(f"  Free      : {total - reserved:.2f} GB  ← headroom for GRPO generations")

In [ ]:
"""
CELL 5 — Quick generation test
Confirms the model and tokenizer are working before you start GRPO training.
"""

test_prompt = (
    "<|im_start|>system\nYou are a math assistant.<|im_end|>\n"
    "<|im_start|>user\nWhat is 12 + 7?<|im_end|>\n"
    "<|im_start|>assistant\n"
)

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,           # greedy for deterministic smoke test
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated tokens (exclude the prompt)
generated = output_ids[0][inputs["input_ids"].shape[1]:]
response  = tokenizer.decode(generated, skip_special_tokens=True)

print(f"Prompt   : What is 12 + 7?")
print(f"Response : {response}")
print("\nSmoke test passed — model is ready for GRPO training.")

In [ ]:
"""
CELL 6 — Google Drive mount + checkpoint helper

Colab T4 sessions disconnect after ~4 hours of inactivity.
Mounting Drive lets you save checkpoints that survive disconnects.
"""


from google.colab import drive
import os

drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/grpo_qwen_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

def save_checkpoint(model, tokenizer, step: int):
    """Call this every N training steps to save adapter weights to Drive."""
    path = os.path.join(CHECKPOINT_DIR, f"step_{step}")
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    print(f"Checkpoint saved → {path}")

# Example: save_checkpoint(model, tokenizer, step=100)

print("""
─────────────────────────────────────────────
Setup complete. Everything is ready:
  ✓ T4 GPU confirmed
  ✓ wandb connected
  ✓ Qwen2.5-1.5B loaded in 4-bit NF4
  ✓ Smoke test passed
  ✓ Drive mounted for checkpoints

Next: run gsm8k_data_prep.py to build your dataset,
then write the reward functions (Phase 3).
─────────────────────────────────────────────
""")

In [ ]:
#-------------------------------------------------------------------------DATASETS LOADING AND PROMPT FORMATING----------------------------------------------------------------------

In [ ]:
import re
from datasets import load_dataset
from transformers import AutoTokenizer

In [ ]:
# ─────────────────────────────────────────────
# 1. Load GSM8K
# ─────────────────────────────────────────────

def load_gsm8k():
    """
    GSM8K: 8,500 grade-school math problems with step-by-step solutions.
    Each example has:
      - 'question': the math problem (str)
      - 'answer':   full solution ending with "#### <number>" (str)
    """
    dataset = load_dataset("openai/gsm8k", "main")
    print(f"Train size : {len(dataset['train'])}")   # 7,473
    print(f"Test size  : {len(dataset['test'])}")    # 1,319
    return dataset

In [ ]:
# ─────────────────────────────────────────────
# 2. Extract numeric ground-truth answer
# ─────────────────────────────────────────────

def extract_ground_truth(answer_text: str) -> str | None:
    """
    GSM8K answers always end with '#### <number>'.
    Extracts and normalises the number so reward comparison is reliable.

    Examples:
      "She has 3 apples.\n#### 3"          -> "3"
      "Total cost is $12.50\n#### 12.5"    -> "12.5"
      "He owes -5 dollars\n#### -5"        -> "-5"
    """
    match = re.search(r"####\s*([-\d,\.]+)", answer_text)
    if not match:
        return None
    # Remove commas used as thousand-separators (e.g. "1,200" -> "1200")
    raw = match.group(1).replace(",", "")
    # Normalise: drop trailing .0 so "3.0" == "3"
    try:
        number = float(raw)
        return str(int(number)) if number == int(number) else str(number)
    except ValueError:
        return raw

In [ ]:
# ─────────────────────────────────────────────
# 3. System & user prompt templates
# ─────────────────────────────────────────────

SYSTEM_PROMPT = """You are a precise math reasoning assistant.

When solving a problem:
1. Think step by step inside <think> tags.
2. Put ONLY the final numeric answer inside <answer> tags.

Format your response EXACTLY like this:
<think>
[your step-by-step reasoning here]
</think>
<answer>[number only]</answer>"""


def format_prompt(question: str, tokenizer) -> str:
    """
    Formats a GSM8K question into the Qwen2.5-Instruct chat template.

    Qwen2.5 uses the ChatML format:
      <|im_start|>system ... <|im_end|>
      <|im_start|>user   ... <|im_end|>
      <|im_start|>assistant

    We apply the tokenizer's built-in chat template so the prompt is
    100% consistent with how the base model was instruction-tuned.
    The 'add_generation_prompt=True' leaves the assistant turn open,
    which is what GRPOTrainer expects to continue from.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question.strip()},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,   # appends <|im_start|>assistant\n
    )
    return prompt

In [ ]:
# ─────────────────────────────────────────────
# 4. Build the formatted dataset
# ─────────────────────────────────────────────

def build_dataset(tokenizer, split: str = "train", max_samples: int = None):
    """
    Returns a list of dicts ready for GRPOTrainer:
      {
        "prompt"       : formatted chat string (input to the model),
        "ground_truth" : normalised numeric answer (used by reward fn),
        "question"     : raw question text (for logging/debugging),
      }

    GRPOTrainer expects the dataset to have a 'prompt' column.
    We carry 'ground_truth' through so the reward function can access it.

    Args:
        tokenizer   : loaded Qwen2.5 tokenizer
        split       : "train" or "test"
        max_samples : cap dataset size (useful for quick debug runs)
    """
    raw = load_gsm8k()[split]

    if max_samples is not None:
        raw = raw.select(range(min(max_samples, len(raw))))

    formatted = []
    skipped = 0

    for example in raw:
        gt = extract_ground_truth(example["answer"])
        if gt is None:
            skipped += 1
            continue

        formatted.append({
            "prompt":        format_prompt(example["question"], tokenizer),
            "ground_truth":  gt,
            "question":      example["question"],
        })

    print(f"Built {len(formatted)} examples from '{split}' split "
          f"({skipped} skipped — no parseable answer).")
    return formatted

In [ ]:
# ─────────────────────────────────────────────
# 5. Answer extractor for the reward function
# ─────────────────────────────────────────────

def extract_model_answer(completion: str) -> str | None:
    """
    Extracts the number from the model's <answer> tag.
    Used inside the reward function to compare against ground_truth.

    Handles common model output variations:
      <answer>42</answer>               -> "42"
      <answer> 42 </answer>             -> "42"
      <answer>42.0</answer>             -> "42"
      <answer>1,200</answer>            -> "1200"
      <answer>-5</answer>               -> "-5"
      ... text ... <answer>7</answer>   -> "7"  (ignores preamble)

    Returns None if no valid <answer> tag is found.
    """
    match = re.search(r"<answer>\s*([-\d,\.]+)\s*</answer>", completion)
    if not match:
        return None
    raw = match.group(1).replace(",", "")
    try:
        number = float(raw)
        return str(int(number)) if number == int(number) else str(number)
    except ValueError:
        return None


def answers_match(model_answer: str | None, ground_truth: str) -> bool:
    """
    Compares extracted model answer against ground truth.
    Normalises both to float before comparing to handle "3" == "3.0".
    """
    if model_answer is None:
        return False
    try:
        return abs(float(model_answer) - float(ground_truth)) < 1e-6
    except ValueError:
        return model_answer.strip() == ground_truth.strip()

def has_correct_format(completion: str) -> bool:
    return bool(re.search(
        r"<think>.*?</think>.*?<answer>[-\d,\.]+</answer>",
        completion, re.DOTALL
    ))

In [ ]:
# ─────────────────────────────────────────────
# 6. Quick sanity check (run this before training)
# ─────────────────────────────────────────────

def run_sanity_check(tokenizer, n: int = 5):
    """
    Loads n examples and prints prompts + ground truth.
    Run this to confirm everything looks right before starting GRPO.
    """
    dataset = build_dataset(tokenizer, split="train", max_samples=n)

    for i, ex in enumerate(dataset):
        print(f"\n{'='*60}")
        print(f"Example {i+1}")
        print(f"{'='*60}")
        print(f"QUESTION:\n{ex['question']}\n")
        print(f"GROUND TRUTH: {ex['ground_truth']}")
        print(f"\nFORMATTED PROMPT (last 300 chars):\n...{ex['prompt'][-300:]}")

    # Test the extractor on a fake completion
    fake_completions = [
        "<think>7 * 6 = 42</think><answer>42</answer>",
        "<think>Let me calculate...</think><answer>1,200</answer>",
        "<think>Hmm</think><answer>3.0</answer>",
        "I think the answer is 5",       # missing tags — should return None
        "<answer>-15</answer>",
    ]
    print(f"\n{'='*60}")
    print("EXTRACTOR TEST")
    print(f"{'='*60}")
    for fc in fake_completions:
        extracted = extract_model_answer(fc)
        print(f"  Input   : {fc[:60]}")
        print(f"  Extracted: {extracted}\n")

In [ ]:
# ─────────────────────────────────────────────
# 7. Main — load tokenizer and build datasets
# ─────────────────────────────────────────────

if __name__ == "__main__":
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token   # required for batch generation

    # Sanity check first
    run_sanity_check(tokenizer, n=3)

    # Build full train + test datasets
    train_data = build_dataset(tokenizer, split="train")
    test_data  = build_dataset(tokenizer, split="test")

    print(f"\nDatasets ready.")
    print(f"  train_data : {len(train_data)} examples")
    print(f"  test_data  : {len(test_data)} examples")
    print("\nSample keys:", list(train_data[0].keys()))
    print("\nNext step: pass train_data to GRPOTrainer with your reward_fn.")

In [ ]:
#-----------------------------------------------------------------REWARD FUNCTIONS FOR GRPO MATH TRAINING-------------------------------------------------------------

In [ ]:
import re
import math
from typing import Optional
import wandb

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Reward 1 — Correctness  (weight: 1.0)
# ─────────────────────────────────────────────────────────────────────────────

def correctness_reward(
    completions: list[str],
    ground_truths: list[str],
) -> list[float]:
    """
    The primary training signal. Binary: correct = 1.0, wrong = 0.0.

    Why binary and not partial credit?
    ─────────────────────────────────
    Partial credit (e.g. reward proportional to how close the number is)
    causes the model to learn to produce numbers near the answer rather
    than to reason correctly. Binary reward forces genuine understanding.
    "Almost right" in math is still wrong.

    Why not -1.0 for wrong answers?
    ────────────────────────────────
    GRPO already normalises rewards within each group using the group mean
    as a baseline. A wrong answer in a group where others are also wrong
    will still produce a negative advantage. Explicit negative reward
    can destabilise early training when the model is mostly wrong.
    """
    rewards = []
    for completion, gt in zip(completions, ground_truths):
        extracted = extract_model_answer(completion)
        reward = 1.0 if answers_match(extracted, gt) else 0.0
        rewards.append(reward)
    return rewards


# ─────────────────────────────────────────────────────────────────────────────
# Reward 2 — Format  (weight: 0.1)
# ─────────────────────────────────────────────────────────────────────────────

def format_reward(completions: list[str]) -> list[float]:
    """
    Rewards the model for using the correct <think>...</think><answer>...</answer>
    structure, in the right order.

    Why do we need this at all?
    ───────────────────────────
    Without format reward, the model quickly learns a shortcut:
    skip all reasoning, output <answer>42</answer>, and occasionally
    get lucky. This collapses the chain-of-thought entirely.
    The format reward keeps the reasoning scaffold alive long enough
    for the correctness reward to reinforce it.

    Why 0.1 and not higher?
    ────────────────────────
    If format reward > 0.2 × correctness reward, the model optimises
    for well-formatted wrong answers. The 0.1 ceiling keeps format as
    a soft constraint, never the dominant training signal.

    Checks (all must pass for full reward):
      ✓ <think> tag present
      ✓ </think> tag present
      ✓ <answer> tag present
      ✓ </answer> tag present
      ✓ <think> appears before <answer>
      ✓ <answer> contains at least one digit
    """
    # Pre-compiled patterns — faster than re.search inside a loop
    THINK_OPEN   = re.compile(r"<think>",              re.DOTALL)
    THINK_CLOSE  = re.compile(r"</think>",             re.DOTALL)
    ANSWER_BLOCK = re.compile(
        r"<answer>\s*[-\d,\.]+\s*</answer>",           re.DOTALL
    )
    ORDER_CHECK  = re.compile(
        r"<think>.*?</think>.*?<answer>.*?</answer>",  re.DOTALL
    )

    rewards = []
    for completion in completions:
        score = 0.0

        has_think_open  = bool(THINK_OPEN.search(completion))
        has_think_close = bool(THINK_CLOSE.search(completion))
        has_answer      = bool(ANSWER_BLOCK.search(completion))
        correct_order   = bool(ORDER_CHECK.search(completion))

        if has_think_open and has_think_close and has_answer and correct_order:
            score = 0.1   # full format reward
        elif has_think_open or has_answer:
            score = 0.03  # partial: something is there, just malformed

        rewards.append(score)
    return rewards


# ─────────────────────────────────────────────────────────────────────────────
# Reward 3 — Length penalty  (optional, weight: up to -0.05)
# ─────────────────────────────────────────────────────────────────────────────

def length_penalty_reward(
    completions: list[str],
    target_tokens: int = 300,
    max_tokens: int    = 500,
    tokenizer          = None,
) -> list[float]:
    """
    Applies a mild negative penalty for reasoning chains that are
    excessively long, to prevent the model padding its <think> block
    to game the format reward or to "look like it's reasoning more."

    Penalty curve (smooth, not cliff-edge):
      - Completions ≤ target_tokens  : 0.0   (no penalty)
      - target_tokens < len ≤ max    : linear ramp from 0.0 to -0.05
      - len > max_tokens             : -0.05  (capped)

    Why so mild?
    ────────────
    Some problems genuinely need long reasoning (multi-step word problems).
    A harsh length penalty would punish correct long reasoning alongside
    incorrect padding. The 0.05 cap means even the worst padding cannot
    outweigh a single correctness reward (1.0).

    Why measure the <think> block only?
    ─────────────────────────────────────
    Penalising total length would penalise the <answer> tag too.
    We only want to discourage padding inside the reasoning section.

    Args:
        tokenizer: if provided, counts in tokens (more accurate).
                   if None, falls back to word count (fast approximation).
    """
    def measure_think_length(text: str) -> int:
        match = re.search(r"<think>(.*?)</think>", text, re.DOTALL)
        if not match:
            return 0
        think_content = match.group(1)
        if tokenizer is not None:
            return len(tokenizer.encode(think_content, add_special_tokens=False))
        else:
            return len(think_content.split())  # word count fallback

    rewards = []
    for completion in completions:
        length = measure_think_length(completion)
        if length <= target_tokens:
            penalty = 0.0
        elif length <= max_tokens:
            # Linear ramp: 0.0 at target_tokens, -0.05 at max_tokens
            excess = length - target_tokens
            ramp   = max_tokens - target_tokens
            penalty = -0.05 * (excess / ramp)
        else:
            penalty = -0.05   # cap
        rewards.append(penalty)
    return rewards


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Combined reward_fn — this is what GRPOTrainer calls
# ─────────────────────────────────────────────────────────────────────────────

# Controls — flip USE_LENGTH_PENALTY off if you want faster, simpler training
USE_LENGTH_PENALTY = True

# Global step counter for wandb logging (GRPOTrainer doesn't pass step to reward_fn)
_reward_step = 0


def reward_fn(completions: list[str], **kwargs) -> list[float]:
    """
    Combined reward function with wandb logging.

    GRPOTrainer signature:
        reward_fn(completions, **kwargs) -> list[float]

    'ground_truth' is carried from your dataset as a kwarg because you
    added it as a column in build_dataset(). TRL passes all extra
    dataset columns as kwargs automatically.

    Total reward per completion:
        correctness  × 1.0   (0.0 or 1.0)
      + format                (0.0, 0.03, or 0.1)
      + length_penalty        (0.0 to -0.05)   [if enabled]
    ─────────────────────────────────────────
    Range: [-0.05, 1.1]

    The dominant signal is always correctness. Format and length
    are soft regularisers that shape the output structure without
    ever overriding the primary objective.
    """
    global _reward_step
    _reward_step += 1

    ground_truths = kwargs["ground_truth"]   # list[str], one per completion

    # ── Compute each component ────────────────────────────────────────────────
    correctness = correctness_reward(completions, ground_truths)
    fmt         = format_reward(completions)
    length_pen  = (
        length_penalty_reward(completions, tokenizer=tokenizer)
        if USE_LENGTH_PENALTY else
        [0.0] * len(completions)
    )

    # ── Combine ───────────────────────────────────────────────────────────────
    total = [
        c + f + l
        for c, f, l in zip(correctness, fmt, length_pen)
    ]

    # ── wandb logging (every 10 reward calls to avoid spam) ──────────────────
    if _reward_step % 10 == 0:
        n = len(completions)
        wandb.log({
            "reward/correctness"    : sum(correctness) / n,
            "reward/format"         : sum(fmt)         / n,
            "reward/length_penalty" : sum(length_pen)  / n,
            "reward/total"          : sum(total)        / n,
            "reward/correct_count"  : sum(1 for c in correctness if c > 0),
            "reward/format_count"   : sum(1 for f in fmt if f >= 0.1),
            "reward_step"           : _reward_step,
        })

    return total

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Sanity check — run this before plugging into GRPOTrainer
# ─────────────────────────────────────────────────────────────────────────────

def test_reward_functions():
    """
    Tests all reward components against known inputs.
    Every assertion must pass before you start training.
    """
    print("Running reward function tests...\n")

    # ── Test cases: (completion, ground_truth, expected_correct) ──────────────
    cases = [
        # Perfect response
        (
            "<think>\n3 bags × 7 apples = 21 apples\n21 + 5 = 26\n</think>\n<answer>26</answer>",
            "26",
            True,
        ),
        # Correct number, wrong format (no tags)
        (
            "The answer is 26.",
            "26",
            False,
        ),
        # Wrong answer, correct format
        (
            "<think>3 × 7 = 21</think><answer>21</answer>",
            "26",
            False,
        ),
        # Correct with thousands comma
        (
            "<think>1000 + 200 = 1200</think><answer>1,200</answer>",
            "1200",
            True,
        ),
        # Correct with float normalisation
        (
            "<think>10 / 4 = 2.5</think><answer>2.5</answer>",
            "2.5",
            True,
        ),
        # Correct with .0 normalisation
        (
            "<think>6 / 2 = 3.0</think><answer>3.0</answer>",
            "3",
            True,
        ),
        # Negative answer
        (
            "<think>5 - 20 = -15</think><answer>-15</answer>",
            "-15",
            True,
        ),
        # Wrong order: answer before think
        (
            "<answer>26</answer><think>reasoning after</think>",
            "26",
            True,   # correctness = True (number is right)
            # but format will be partial because order is wrong
        ),
    ]

    all_passed = True
    for i, (completion, gt, expected_correct) in enumerate(cases):
        extracted  = extract_model_answer(completion)
        got_correct = answers_match(extracted, gt)
        fmt_score   = format_reward([completion])[0]
        len_score   = length_penalty_reward([completion])[0]

        status = "✓" if got_correct == expected_correct else "✗ FAIL"
        if got_correct != expected_correct:
            all_passed = False

        print(f"  [{status}] Case {i+1}")
        print(f"        completion    : {completion[:60]}...")
        print(f"        ground_truth  : {gt}")
        print(f"        extracted     : {extracted}")
        print(f"        correct       : {got_correct}  (expected {expected_correct})")
        print(f"        format score  : {fmt_score}")
        print(f"        length penalty: {len_score}")
        print()

    # ── Combined reward test ───────────────────────────────────────────────────
    print("Testing combined reward_fn signature...")
    completions   = [c[0] for c in cases[:4]]
    ground_truths = [c[1] for c in cases[:4]]
    rewards = reward_fn(completions, ground_truth=ground_truths)

    assert len(rewards) == 4, "reward_fn must return one value per completion"
    assert all(isinstance(r, float) for r in rewards), "All rewards must be floats"
    assert rewards[0] > rewards[1], "Perfect response must beat no-tags response"
    assert rewards[0] > rewards[2], "Correct answer must beat wrong answer"

    print(f"  Combined rewards: {[round(r, 3) for r in rewards]}")
    print(f"\n{'All tests passed ✓' if all_passed else 'Some tests FAILED ✗ — fix before training'}")
    return all_passed


# ── Run tests ─────────────────────────────────────────────────────────────────
if __name__ == "__main__" or True:
    tests_ok = test_reward_functions()

    if tests_ok:
        print("""
─────────────────────────────────────────────────────────────
Reward functions ready. Pass reward_fn to GRPOTrainer like:

    from trl import GRPOTrainer, GRPOConfig

    trainer = GRPOTrainer(
        model        = model,
        tokenizer    = tokenizer,
        reward_funcs = [reward_fn],     # ← your combined reward
        args         = GRPOConfig(...),
        train_dataset= train_data,      # from build_dataset()
    )
    trainer.train()

─────────────────────────────────────────────────────────────
""")


In [ ]:
#-------------------------------------------------------------------------GRPOTRAINER TRAINING LOOP CODE----------------------------------------------------------------------

In [ ]:
import os
import torch
import wandb
from datasets import Dataset
from transformers import TrainerCallback
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
MODEL_NAME      = "Qwen/Qwen2.5-1.5B-Instruct"
CHECKPOINT_DIR  = "/content/drive/MyDrive/grpo_qwen_checkpoints"
TOTAL_STEPS     = 500       # ~1.5–2 hrs on T4. Raise to 1000 if you have Colab Pro.
SAVE_EVERY      = 100       # save adapter weights to Drive every N steps
LOG_EVERY       = 5         # log metrics to wandb every N steps
GRAD_ACCUM      = 8         # effective batch = 1 × 8 = 8 prompts per update
NUM_GENERATIONS = 4         # G in GRPO: completions sampled per prompt
MAX_PROMPT_LEN  = 256       # tokens — GSM8K questions are short
MAX_NEW_TOKENS  = 300       # tokens for the model's reasoning + answer

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 1 — Attach LoRA adapters
# ─────────────────────────────────────────────────────────────────────────────
# We train LoRA adapters on top of the frozen 4-bit base model.
# This is the only part of the model that accumulates gradients.
#
# Why LoRA for GRPO?
#   Full fine-tuning a 1.5B model needs ~12 GB of optimizer states alone
#   — impossible on a T4 with GRPO's generation overhead.
#   LoRA adds ~2M trainable parameters (r=16) vs 1.5B base = 0.13% of weights.
#   Adapter weights are fp32; base weights stay int4. Fits comfortably.
#
# Target modules: the Q, K, V, O projections of every attention layer.
# These are where the model's reasoning behaviour is encoded.
# Adding gate/up/down proj (MLP layers) improves capacity but costs ~1 GB more.

print("Attaching LoRA adapters...")

lora_config = LoraConfig(
    task_type       = TaskType.CAUSAL_LM,
    r               = 16,           # rank — higher = more capacity, more VRAM
    lora_alpha      = 32,           # scaling factor; convention: 2 × r
    lora_dropout    = 0.05,         # light dropout to reduce overfitting
    bias            = "none",       # don't train bias terms — saves memory
    target_modules  = [             # attention projections only (safe for T4)
        "q_proj", "k_proj",
        "v_proj", "o_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: trainable params: ~2.6M || all params: ~1.54B || 0.17%

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 2 — Truncate prompts at the dataset level (replaces max_prompt_length)
# ─────────────────────────────────────────────────────────────────────────────
# In current TRL, prompt length control is your responsibility before training.
# We tokenize each prompt, check its length, and drop anything over the limit.
# GSM8K prompts are short (~80–120 tokens) so almost nothing gets dropped.

def truncate_and_convert(data: list, tokenizer, max_tokens: int) -> Dataset:
    """
    Converts list of dicts to HuggingFace Dataset.
    Drops examples where the prompt exceeds max_tokens.
    This replaces the old max_prompt_length GRPOConfig argument.
    """
    filtered = []
    dropped  = 0
    for ex in data:
        token_len = len(tokenizer(ex["prompt"], add_special_tokens=False).input_ids)
        if token_len <= max_tokens:
            filtered.append(ex)
        else:
            dropped += 1

    if dropped:
        print(f"  Dropped {dropped} examples with prompt > {max_tokens} tokens.")

    ds = Dataset.from_list(filtered)
    # GRPOTrainer only needs 'prompt' + any reward kwargs ('ground_truth')
    # Drop 'question' — it was for debugging only
    if "question" in ds.column_names:
        ds = ds.remove_columns(["question"])
    return ds

print("Preparing datasets...")
hf_train = truncate_and_convert(train_data, tokenizer, MAX_PROMPT_LEN)
hf_test  = truncate_and_convert(test_data,  tokenizer, MAX_PROMPT_LEN)
print(f"  Train: {len(hf_train)} examples | columns: {hf_train.column_names}")
print(f"  Test : {len(hf_test)}  examples | columns: {hf_test.column_names}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 3 — GRPOConfig (FIXED for current TRL)
# ─────────────────────────────────────────────────────────────────────────────
# REMOVED:  max_prompt_length   ← was the source of the TypeError
# REMOVED:  eval_strategy       ← now called evaluation_strategy in some versions;
#                                  omit entirely and use save_steps for checkpoints
# KEPT:     max_completion_length, num_generations, all standard training args

grpo_config = GRPOConfig(

    # ── Output & logging ──────────────────────────────────────────────────────
    output_dir    = CHECKPOINT_DIR,
    run_name      = "qwen2.5-1.5b-gsm8k-grpo",
    report_to     = "wandb",
    logging_steps = LOG_EVERY,

    # ── Training duration ─────────────────────────────────────────────────────
    max_steps     = TOTAL_STEPS,

    # ── Batch sizes ───────────────────────────────────────────────────────────
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = GRAD_ACCUM,

    # ── GRPO-specific ─────────────────────────────────────────────────────────
    num_generations       = NUM_GENERATIONS,
    max_completion_length = MAX_NEW_TOKENS,

    # ── Optimiser ─────────────────────────────────────────────────────────────
    learning_rate     = 5e-6,
    lr_scheduler_type = "cosine",
    warmup_steps      = 20,
    optim             = "adamw_torch_fused",

    # ── Precision & memory ────────────────────────────────────────────────────
    bf16                   = True,
    gradient_checkpointing = True,

    # ── KL penalty — renamed from kl_coef to beta ────────────────────────────
    beta = 0.001,
    # beta=0.0 (default): reference model not loaded at all — saves ~1 GB VRAM
    # beta=0.001: light KL penalty, matches DeepSeek-R1 small model setting
    # For T4 with tight VRAM, you can leave beta=0.0 entirely and skip this line

    # ── Checkpointing ─────────────────────────────────────────────────────────
    save_steps            = SAVE_EVERY,
    save_total_limit      = 3,
    remove_unused_columns = False,
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 4 — wandb callback (unchanged from original)
# ─────────────────────────────────────────────────────────────────────────────

class GRPOWandbCallback(TrainerCallback):
    def __init__(self, eval_dataset, tokenizer, model, log_every_n_steps=100):
        self.eval_dataset      = eval_dataset
        self.tokenizer         = tokenizer
        self.model             = model
        self.log_every_n_steps = log_every_n_steps
        n = min(3, len(eval_dataset))
        self.fixed_prompts = [eval_dataset[i]["prompt"]        for i in range(n)]
        self.fixed_gts     = [eval_dataset[i]["ground_truth"]  for i in range(n)]

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.log_every_n_steps != 0:
            return

        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved()  / 1e9
        wandb.log({
            "system/vram_allocated_gb": allocated,
            "system/vram_reserved_gb" : reserved,
            "train/step"              : state.global_step,
        })

        self.model.eval()
        table_rows = []
        for prompt, gt in zip(self.fixed_prompts, self.fixed_gts):
            inputs = self.tokenizer(
                prompt, return_tensors="pt", truncation=True,
                max_length=MAX_PROMPT_LEN,
            ).to(self.model.device)
            with torch.no_grad():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens = 200,
                    do_sample      = True,
                    temperature    = 0.7,
                    top_p          = 0.9,
                    pad_token_id   = self.tokenizer.eos_token_id,
                )
            generated  = output_ids[0][inputs["input_ids"].shape[1]:]
            completion = self.tokenizer.decode(generated, skip_special_tokens=True)
            extracted  = extract_model_answer(completion)
            correct    = answers_match(extracted, gt)
            table_rows.append([
                state.global_step, gt,
                extracted or "—",
                "✓" if correct else "✗",
                completion[:300],
            ])
        wandb.log({
            "samples/completions": wandb.Table(
                columns=["step", "ground_truth", "extracted", "correct", "completion"],
                data=table_rows,
            )
        })
        self.model.train()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 5 — GRPOTrainer
# ─────────────────────────────────────────────────────────────────────────────

print("Initialising GRPOTrainer...")
trainer = GRPOTrainer(
    model         = model,
    processing_class  = tokenizer,
    reward_funcs  = [reward_fn],
    args          = grpo_config,
    train_dataset = hf_train,
    eval_dataset  = hf_test,
)
trainer.add_callback(
    GRPOWandbCallback(
        eval_dataset      = hf_test,
        tokenizer         = tokenizer,
        model             = model,
        log_every_n_steps = SAVE_EVERY,
    )
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 6 — VRAM check + train
# ─────────────────────────────────────────────────────────────────────────────

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nVRAM before training:")
print(f"  Allocated : {allocated:.2f} GB")
print(f"  Free      : {total - reserved:.2f} GB")
if (total - reserved) < 2.0:
    print("  ⚠ Low headroom — reduce MAX_NEW_TOKENS to 200 if you OOM.")

print(f"\nStarting training ({TOTAL_STEPS} steps)...")
print(f"Watch live at: {wandb.run.url}\n")

try:
    trainer.train(resume_from_checkpoint=True)
except KeyboardInterrupt:
    step = trainer.state.global_step
    path = os.path.join(CHECKPOINT_DIR, f"interrupted_step_{step}")
    trainer.save_model(path)
    tokenizer.save_pretrained(path)
    print(f"Interrupted — saved to {path}")
except torch.cuda.OutOfMemoryError:
    print("\n⚠ OOM. Try in order:")
    print("  1. Reduce MAX_NEW_TOKENS: 300 → 200")
    print("  2. Reduce NUM_GENERATIONS: 4 → 2")
    print("  3. Runtime → Restart → re-run all cells")
    raise

# Save
final_path = os.path.join(CHECKPOINT_DIR, "final")
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f"Saved to {final_path}")
wandb.finish()

In [ ]:
#---------------------------------------------------------------------------------EVALUATION--------------------------------------------------------------------------------------

In [ ]:
"""
CELL 9 — Evaluation: Base vs GRPO Fine-tuned on GSM8K Test Set
================================================================

"""

import os
import re
import json
import time
import torch
import wandb
import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────

MODEL_NAME      = "Qwen/Qwen2.5-1.5B-Instruct"
CHECKPOINT_DIR  = "/content/drive/MyDrive/grpo_qwen_checkpoints"
ADAPTER_PATH    = os.path.join(CHECKPOINT_DIR, "final")   # or "step_500"
RESULTS_PATH    = os.path.join(CHECKPOINT_DIR, "eval_results.json")

MAX_NEW_TOKENS  = 300
MAX_PROMPT_LEN  = 256
EVAL_SUBSET     = 200   # set to e.g. 200 for a quick smoke-test; None = full 1319

# Generation settings — greedy for deterministic, reproducible evaluation
GEN_KWARGS = dict(
    max_new_tokens = MAX_NEW_TOKENS,
    do_sample      = False,          # greedy decoding
    temperature    = 1.0,
    pad_token_id   = None,           # set after tokenizer loads
)

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load test set
# ─────────────────────────────────────────────────────────────────────────────

print("Loading GSM8K test set...")
raw_test = load_dataset("openai/gsm8k", "main")["test"]
if EVAL_SUBSET:
    raw_test = raw_test.select(range(EVAL_SUBSET))

test_examples = []
for ex in raw_test:
    gt = extract_ground_truth(ex["answer"])
    if gt is not None:
        test_examples.append({"question": ex["question"], "ground_truth": gt})

print(f"Evaluating on {len(test_examples)} examples.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Core eval function
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_model(model, tokenizer, examples: list, label: str) -> dict:
    """
    Runs model on all examples, returns a results dict with:
      - per-example predictions
      - aggregate accuracy, format rate, avg reasoning length
      - timing info
    """
    GEN_KWARGS["pad_token_id"] = tokenizer.eos_token_id
    model.eval()

    results = []
    start   = time.time()

    for ex in tqdm(examples, desc=f"Evaluating {label}"):
        prompt = format_prompt(ex["question"], tokenizer)

        inputs = tokenizer(
            prompt,
            return_tensors  = "pt",
            truncation      = True,
            max_length      = MAX_PROMPT_LEN,
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(**inputs, **GEN_KWARGS)

        generated  = output_ids[0][inputs["input_ids"].shape[1]:]
        completion = tokenizer.decode(generated, skip_special_tokens=True)

        extracted = extract_model_answer(completion)
        correct   = answers_match(extracted, ex["ground_truth"])
        formatted = has_correct_format(completion)

        # Measure reasoning length (words inside <think> block)
        think_match = re.search(r"<think>(.*?)</think>", completion, re.DOTALL)
        think_words = len(think_match.group(1).split()) if think_match else 0

        results.append({
            "question"      : ex["question"],
            "ground_truth"  : ex["ground_truth"],
            "completion"    : completion,
            "extracted"     : extracted,
            "correct"       : correct,
            "formatted"     : formatted,
            "think_words"   : think_words,
        })

    elapsed = time.time() - start

    # ── Aggregate metrics ─────────────────────────────────────────────────────
    n          = len(results)
    n_correct  = sum(r["correct"]   for r in results)
    n_format   = sum(r["formatted"] for r in results)
    think_lens = [r["think_words"]  for r in results]

    return {
        "label"            : label,
        "n_examples"       : n,
        "accuracy"         : n_correct / n,
        "format_rate"      : n_format  / n,
        "avg_think_words"  : float(np.mean(think_lens)),
        "median_think_words": float(np.median(think_lens)),
        "elapsed_mins"     : elapsed / 60,
        "per_example"      : results,
    }


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load and evaluate BASE model
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "═"*55)
print("EVALUATING: Base model (no fine-tuning)")
print("═"*55)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = BNB_CONFIG,
    device_map          = "auto",
    torch_dtype         = torch.bfloat16,
    trust_remote_code   = True,
)
base_model.config.use_cache = True   # enable KV cache for faster inference

base_results = evaluate_model(base_model, tokenizer, test_examples, "Base")

# Free base model VRAM before loading fine-tuned
del base_model
torch.cuda.empty_cache()
print(f"\nBase model eval complete. VRAM freed.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load and evaluate FINE-TUNED model
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "═"*55)
print("EVALUATING: GRPO fine-tuned model")
print("═"*55)

if not os.path.exists(ADAPTER_PATH):
    raise FileNotFoundError(
        f"Adapter not found at {ADAPTER_PATH}. "
        "Check CHECKPOINT_DIR and ADAPTER_PATH at the top of this cell."
    )

ft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = BNB_CONFIG,
    device_map          = "auto",
    torch_dtype         = torch.bfloat16,
    trust_remote_code   = True,
)
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_PATH)
ft_model.config.use_cache = True

ft_results = evaluate_model(ft_model, tokenizer, test_examples, "GRPO fine-tuned")

del ft_model
torch.cuda.empty_cache()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Results table
# ─────────────────────────────────────────────────────────────────────────────

def pct(x): return f"{x*100:.1f}%"
def delta(ft, base):
    d = (ft - base) * 100
    sign = "+" if d >= 0 else ""
    return f"({sign}{d:.1f}pp)"

acc_base  = base_results["accuracy"]
acc_ft    = ft_results["accuracy"]
fmt_base  = base_results["format_rate"]
fmt_ft    = ft_results["format_rate"]
tw_base   = base_results["avg_think_words"]
tw_ft     = ft_results["avg_think_words"]

print(f"""
╔{'═'*57}╗
║{'EVALUATION RESULTS — GSM8K TEST SET':^57}║
╠{'═'*57}╣
║ {'Metric':<30} {'Base':>8}  {'Fine-tuned':>10}  {'Δ':>6} ║
╠{'═'*57}╣
║ {'Accuracy (% correct)':<30} {pct(acc_base):>8}  {pct(acc_ft):>10}  {delta(acc_ft, acc_base):>6} ║
║ {'Format rate (% with tags)':<30} {pct(fmt_base):>8}  {pct(fmt_ft):>10}  {delta(fmt_ft, fmt_base):>6} ║
║ {'Avg reasoning length (words)':<30} {tw_base:>8.0f}  {tw_ft:>10.0f}  {'':>6} ║
║ {'Examples evaluated':<30} {base_results['n_examples']:>8}  {ft_results['n_examples']:>10}  {'':>6} ║
║ {'Eval time (mins)':<30} {base_results['elapsed_mins']:>8.1f}  {ft_results['elapsed_mins']:>10.1f}  {'':>6} ║
╚{'═'*57}╝
""")



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Qualitative examples
# ─────────────────────────────────────────────────────────────────────────────
# Find cases where base was wrong but fine-tuned was right — most compelling.

def find_interesting_cases(base_per, ft_per, n=3):
    """Returns cases where FT succeeded and base failed."""
    wins = [
        (b, f) for b, f in zip(base_per, ft_per)
        if not b["correct"] and f["correct"]
    ]
    return wins[:n]

wins = find_interesting_cases(
    base_results["per_example"],
    ft_results["per_example"],
)

print(f"── Qualitative examples: fine-tuned correct, base wrong ─────")
for i, (base_ex, ft_ex) in enumerate(wins):
    print(f"\n{'─'*55}")
    print(f"Example {i+1}")
    print(f"Question     : {base_ex['question'][:120]}...")
    print(f"Ground truth : {base_ex['ground_truth']}")
    print(f"\n[Base model]")
    print(f"  Extracted  : {base_ex['extracted'] or '(none)'}")
    print(f"  Formatted  : {'Yes' if base_ex['formatted'] else 'No'}")
    print(f"  Completion : {base_ex['completion'][:200]}...")
    print(f"\n[Fine-tuned]")
    print(f"  Extracted  : {ft_ex['extracted']}")
    print(f"  Formatted  : {'Yes' if ft_ex['formatted'] else 'Yes'}")
    print(f"  Completion : {ft_ex['completion'][:300]}...")



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Log to wandb
# ─────────────────────────────────────────────────────────────────────────────

wandb.init(
    project = "grpo-math-qwen",
    name    = "evaluation-base-vs-finetuned",
    config  = {
        "model"        : MODEL_NAME,
        "adapter_path" : ADAPTER_PATH,
        "n_examples"   : len(test_examples),
        "decoding"     : "greedy",
    },
)

# Summary metrics
wandb.log({
    "eval/base_accuracy"       : acc_base,
    "eval/ft_accuracy"         : acc_ft,
    "eval/accuracy_delta"      : acc_ft - acc_base,
    "eval/base_format_rate"    : fmt_base,
    "eval/ft_format_rate"      : fmt_ft,
    "eval/base_avg_think_words": tw_base,
    "eval/ft_avg_think_words"  : tw_ft,
})

# Comparison bar chart (renders natively in wandb)
wandb.log({
    "eval/accuracy_comparison": wandb.plot.bar(
        wandb.Table(
            columns = ["Model", "Accuracy"],
            data    = [
                ["Base Qwen2.5-1.5B",    acc_base],
                ["GRPO fine-tuned",       acc_ft],
            ],
        ),
        "Model", "Accuracy",
        title="GSM8K Accuracy: Base vs GRPO Fine-tuned",
    )
})

# Qualitative wins table
if wins:
    win_rows = [
        [
            b["question"][:100],
            b["ground_truth"],
            b["extracted"] or "—",
            f["extracted"],
            f["completion"][:300],
        ]
        for b, f in wins
    ]
    wandb.log({
        "eval/qualitative_wins": wandb.Table(
            columns = ["question", "ground_truth", "base_answer", "ft_answer", "ft_reasoning"],
            data    = win_rows,
        )
    })

wandb.finish()
print("Results logged to wandb.")



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Save results JSON to Drive
# ─────────────────────────────────────────────────────────────────────────────

# Strip completions from per_example to keep file small, then save
def slim(results):
    return {
        k: v for k, v in results.items()
        if k != "per_example"
    } | {
        "per_example": [
            {kk: vv for kk, vv in ex.items() if kk != "completion"}
            for ex in results["per_example"]
        ]
    }

with open(RESULTS_PATH, "w") as f:
    json.dump({"base": slim(base_results), "finetuned": slim(ft_results)}, f, indent=2)

print(f"Results saved to {RESULTS_PATH}")

print(f"""
{'═'*55}
Evaluation complete.

  Base accuracy     : {pct(acc_base)}
  Fine-tuned acc    : {pct(acc_ft)}  {delta(acc_ft, acc_base)}
  Format rate delta : {delta(fmt_ft, fmt_base)}

README checklist:
  ✓ Copy the results table above into your README
  ✓ Export reward/correctness curve from wandb → PNG
  ✓ Export eval/accuracy_comparison chart → PNG
  ✓ Paste 1–2 qualitative win examples into README
  ✓ Add the resume line to your CV
  ✓ Push adapter to HuggingFace Hub (optional)
{'═'*55}
""")


In [ ]:
# Paste in a new Colab cell and run
import re

# Grab 5 examples from your results
for ex in ft_results["per_example"][:5]:
    print(f"Question : {ex['question'][:80]}")
    print(f"GT       : {ex['ground_truth']}")
    print(f"Extracted: {ex['extracted']}")
    print(f"Correct  : {ex['correct']}")
    print(f"Completion (first 300 chars):")
    print(ex['completion'][:300])
    print("─"*50)

In [ ]:
# Also run this
correct_count = sum(1 for ex in ft_results["per_example"] if ex['correct'])
format_count  = sum(1 for ex in ft_results["per_example"] if ex['formatted'])
none_count    = sum(1 for ex in ft_results["per_example"] if ex['extracted'] is None)

print(f"Correct      : {correct_count} / {len(ft_results['per_example'])}")
print(f"Formatted    : {format_count}")
print(f"No answer tag: {none_count}  ← key number")

In [ ]:
# Paste in a new Colab cell
no_tag = [ex for ex in ft_results["per_example"] if ex["extracted"] is None]

print(f"Showing 5 of {len(no_tag)} completions with no <answer> tag:\n")
for ex in no_tag[:5]:
    print(f"GT         : {ex['ground_truth']}")
    print(f"Completion : {ex['completion'][:400]}")
    print("─" * 50)